# Klasifikasi Buah Segar dan Busuk Menggunakan CNN Transfer Learning (MobileNetV2)

**Dataset:** [Fruits fresh and rotten for classification (Kaggle, Sriram R)](https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification)

Notebook ini berisi seluruh alur proyek dan disusun mengikuti urutan **Bagian A** pada format laporan:

| Bagian notebook | Bagian laporan |
|---|---|
| 1–2. Dataset & pemeriksaan kualitas | A.2 Dataset |
| 3. EDA | A.3 EDA |
| 4. Preprocessing | A.4 Preprocessing |
| 5. Model & pelatihan | A.5 Model |
| 6. Evaluasi | A.6 Hasil evaluasi |
| 7–8. Fitur unik (Grad-CAM, confidence threshold + deteksi luar domain) | A.5e |
| 9. Simpan model | untuk aplikasi |
| 10. Teknologi | A.7 Teknologi |

**Cara menjalankan di Kaggle**
1. Panel kanan → *Session options*: **Accelerator = GPU T4 x2** (atau P100), **Internet = On**.
2. Pastikan dataset sudah ditambahkan (*Add Input*).
3. Klik **Run All**. Untuk menyimpan semua output secara permanen, gunakan **Save Version → Save & Run All (Commit)**.

Semua gambar untuk laporan tersimpan di `/kaggle/working/figures/`, semua angka di `/kaggle/working/results/metrics.json`, dan model di `/kaggle/working/model/`. Gunakan angka dari file tersebut agar laporan, kode, dan aplikasi konsisten.

## 0. Setup dan konfigurasi

In [ ]:
import os, io, json, glob, time, hashlib, zipfile, platform, subprocess, warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import PIL
from PIL import Image
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_recall_fscore_support, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
import keras
from keras import layers

warnings.filterwarnings('ignore')

# ===================== KONFIGURASI =====================
SEED          = 42
IMG_SIZE      = 224        # ukuran input MobileNetV2
BATCH_SIZE    = 32
VAL_SIZE      = 0.15       # proporsi validation yang diambil dari folder train
EPOCHS_HEAD   = int(os.environ.get('EPOCHS_HEAD', 15))   # tahap 1: melatih classifier head
EPOCHS_FT     = int(os.environ.get('EPOCHS_FT', 15))     # tahap 2: fine-tuning
LR_HEAD       = 1e-3
LR_FT         = 1e-5
FINE_TUNE_AT  = 100        # layer MobileNetV2 mulai indeks ini yang dibuka saat fine-tuning
DROPOUT       = 0.3
L2_REG        = 1e-4
WEIGHTS       = os.environ.get('WEIGHTS', 'imagenet')
INPUT_ROOT    = os.environ.get('INPUT_ROOT', '/kaggle/input')
OUT_DIR       = os.environ.get('OUT_DIR', '/kaggle/working')
OOD_DIR       = os.environ.get('OOD_DIR', '')   # opsional: folder gambar non-buah untuk uji luar domain
# =======================================================

FIG_DIR   = os.path.join(OUT_DIR, 'figures')
RES_DIR   = os.path.join(OUT_DIR, 'results')
MODEL_DIR = os.path.join(OUT_DIR, 'model')
for d in (FIG_DIR, RES_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE
RESULTS = {'config': dict(seed=SEED, img_size=IMG_SIZE, batch_size=BATCH_SIZE, val_size=VAL_SIZE,
                          epochs_head=EPOCHS_HEAD, epochs_ft=EPOCHS_FT, lr_head=LR_HEAD, lr_ft=LR_FT,
                          fine_tune_at=FINE_TUNE_AT, dropout=DROPOUT, l2=L2_REG, weights=WEIGHTS)}
sns.set_theme(style='whitegrid')

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print('Gambar disimpan:', path)

print('TensorFlow:', tf.__version__, '| Keras:', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Dataset (Laporan A.2)

Dataset ini memiliki folder ganda (`dataset/` dan `dataset/dataset/`) yang isinya sama persis. Kode di bawah otomatis memilih **satu** folder saja agar data tidak terhitung dua kali.

In [ ]:
def find_data_root(base):
    hits = glob.glob(os.path.join(base, '**', 'train', 'freshapples'), recursive=True)
    roots = sorted({os.path.dirname(os.path.dirname(h)) for h in hits}, key=len)
    if not roots:
        raise FileNotFoundError('Folder dataset tidak ditemukan. Pastikan dataset sudah ditambahkan (Add Input).')
    return roots[0]

DATA_ROOT = find_data_root(INPUT_ROOT)
CLASSES = sorted(d for d in os.listdir(os.path.join(DATA_ROOT, 'train'))
                 if os.path.isdir(os.path.join(DATA_ROOT, 'train', d)))
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
DISPLAY_NAMES = {
    'freshapples': 'Apel Segar', 'freshbanana': 'Pisang Segar', 'freshoranges': 'Jeruk Segar',
    'rottenapples': 'Apel Busuk', 'rottenbanana': 'Pisang Busuk', 'rottenoranges': 'Jeruk Busuk',
}
print('DATA_ROOT :', DATA_ROOT)
print('Kelas     :', CLASSES)

rows = []
for split in ['train', 'test']:
    for c in CLASSES:
        folder = os.path.join(DATA_ROOT, split, c)
        for f in sorted(os.listdir(folder)):
            rows.append(dict(path=os.path.join(folder, f), filename=f, split=split, label=c,
                             fruit=c.replace('fresh', '').replace('rotten', ''),
                             condition='segar' if c.startswith('fresh') else 'busuk'))
df = pd.DataFrame(rows)

counts = (df.pivot_table(index='label', columns='split', values='path', aggfunc='count')
            .fillna(0).astype(int)[['train', 'test']])
counts['total'] = counts.sum(axis=1)
counts.loc['TOTAL'] = counts.sum()
print('\nJumlah file per kelas (sebelum pembersihan):')
print(counts.to_string())
RESULTS['dataset_raw'] = {'per_class': counts.drop(index='TOTAL').to_dict(orient='index'),
                          'total_train': int(counts.loc['TOTAL', 'train']),
                          'total_test': int(counts.loc['TOTAL', 'test']),
                          'total': int(counts.loc['TOTAL', 'total'])}

## 2. Pemeriksaan kualitas data

Setiap file dibuka untuk mengecek apakah rusak, lalu dicatat format, resolusi, kecerahan rata-rata, dan *hash* MD5 (untuk mendeteksi gambar duplikat).

In [ ]:
def inspect_image(path):
    info = dict(path=path, valid=False, md5=None, width=np.nan, height=np.nan, mode=None,
                format=None, mean_r=np.nan, mean_g=np.nan, mean_b=np.nan, brightness=np.nan, error=None)
    try:
        with open(path, 'rb') as fh:
            data = fh.read()
        info['md5'] = hashlib.md5(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im:
            im.verify()
        with Image.open(io.BytesIO(data)) as im:
            im.load()
            info.update(width=im.size[0], height=im.size[1], mode=im.mode, format=im.format)
            small = np.asarray(im.convert('RGB').resize((64, 64)), dtype=np.float32)
            info.update(mean_r=small[..., 0].mean(), mean_g=small[..., 1].mean(),
                        mean_b=small[..., 2].mean(), brightness=small.mean())
        info['valid'] = True
    except Exception as e:
        info['error'] = str(e)[:120]
    return info

t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    meta = pd.DataFrame(list(ex.map(inspect_image, df['path'])))
df = df.merge(meta, on='path')
print(f'Selesai memeriksa {len(df)} file dalam {time.time()-t0:.1f} detik')

invalid = df[~df['valid']]
print('File rusak / tidak bisa dibaca:', len(invalid))
if len(invalid):
    print(invalid[['split', 'label', 'filename', 'error']].to_string())
print('\nFormat file :', df['format'].value_counts(dropna=False).to_dict())
print('Mode warna  :', df['mode'].value_counts(dropna=False).to_dict())
print('\nStatistik resolusi (piksel):')
print(df[['width', 'height']].describe().round(1).to_string())

## 3. Exploratory Data Analysis (Laporan A.3)

### 3.1 Distribusi kelas

In [ ]:
v = df[df['valid']].copy()
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
order = CLASSES
sns.countplot(data=v, y='label', hue='split', order=order, ax=ax[0], palette=['#4C72B0', '#DD8452'])
for cont in ax[0].containers:
    ax[0].bar_label(cont, padding=3, fontsize=9)
ax[0].set_title('Jumlah gambar per kelas (train vs test)')
ax[0].set_xlabel('Jumlah gambar'); ax[0].set_ylabel('')

cond = v[v['split'] == 'train'].groupby(['fruit', 'condition']).size().unstack()
cond.plot(kind='bar', ax=ax[1], color=['#C44E52', '#55A868'])
ax[1].set_title('Train: segar vs busuk per jenis buah'); ax[1].set_xlabel(''); ax[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); savefig('01_distribusi_kelas.png'); plt.show()

tr_counts = v[v['split'] == 'train']['label'].value_counts()
imbalance = tr_counts.max() / tr_counts.min()
print('Rasio ketidakseimbangan (kelas terbanyak / tersedikit) pada train:', round(imbalance, 2))
print(tr_counts.to_string())
RESULTS['eda'] = {'imbalance_ratio_train': float(imbalance)}

### 3.2 Contoh sampel per kelas

In [ ]:
N_SHOW = 5
fig, axes = plt.subplots(NUM_CLASSES, N_SHOW, figsize=(N_SHOW * 2.6, NUM_CLASSES * 2.6))
for i, c in enumerate(CLASSES):
    sample = v[(v['split'] == 'train') & (v['label'] == c)].sample(N_SHOW, random_state=SEED)
    for j, (_, r) in enumerate(sample.iterrows()):
        ax = axes[i, j]
        with Image.open(r['path']) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(DISPLAY_NAMES.get(c, c), fontsize=11)
        ax.set_title(f"{int(r['width'])}x{int(r['height'])}", fontsize=8)
plt.suptitle('Contoh gambar per kelas (judul = resolusi asli)', y=1.0)
plt.tight_layout(); savefig('02_contoh_sampel.png'); plt.show()

### 3.3 Variasi resolusi, kecerahan, dan warna

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=v.sample(min(3000, len(v)), random_state=SEED), x='width', y='height',
                hue='label', s=12, alpha=0.6, ax=ax[0])
ax[0].set_title('Sebaran resolusi asli (sampel)'); ax[0].legend(fontsize=7)

sns.boxplot(data=v, x='brightness', y='label', order=CLASSES, ax=ax[1], color='#8DA0CB')
ax[1].set_title('Kecerahan rata-rata per kelas (0-255)'); ax[1].set_ylabel('')

rgb = v.groupby('label')[['mean_r', 'mean_g', 'mean_b']].mean().loc[CLASSES]
rgb.plot(kind='barh', ax=ax[2], color=['#d62728', '#2ca02c', '#1f77b4'])
ax[2].set_title('Rata-rata kanal warna per kelas'); ax[2].set_ylabel('')
plt.tight_layout(); savefig('03_resolusi_kecerahan_warna.png'); plt.show()

v['aspect'] = v['width'] / v['height']
print('Resolusi min :', int(v['width'].min()), 'x', int(v['height'].min()))
print('Resolusi max :', int(v['width'].max()), 'x', int(v['height'].max()))
print('Median       :', int(v['width'].median()), 'x', int(v['height'].median()))
print('Aspect ratio (min/median/max):', round(float(v['aspect'].min()), 2), round(float(v['aspect'].median()), 2), round(float(v['aspect'].max()), 2))
print('\nKecerahan rata-rata per kelas:')
print(v.groupby('label')['brightness'].agg(['mean', 'std', 'min', 'max']).round(1).loc[CLASSES].to_string())
RESULTS['eda'].update({'res_min': [int(v['width'].min()), int(v['height'].min())],
                       'res_max': [int(v['width'].max()), int(v['height'].max())],
                       'res_median': [int(v['width'].median()), int(v['height'].median())]})

### 3.4 Duplikat dan potensi kebocoran data

In [ ]:
train_v, test_v = v[v['split'] == 'train'], v[v['split'] == 'test']
train_md5, test_md5 = set(train_v['md5']), set(test_v['md5'])
label_per_md5 = v.groupby('md5')['label'].nunique()
conflict_md5 = set(label_per_md5[label_per_md5 > 1].index)

dup_stats = {
    'duplikat_di_dalam_train': int(train_v.duplicated('md5').sum()),
    'duplikat_di_dalam_test': int(test_v.duplicated('md5').sum()),
    'gambar_train_yang_juga_ada_di_test': int(train_v['md5'].isin(test_md5).sum()),
    'gambar_dengan_label_berbeda': int(v['md5'].isin(conflict_md5).sum()),
}
for k, val in dup_stats.items():
    print(f'{k:40s}: {val}')
RESULTS['eda']['duplicates'] = dup_stats

### 3.5 Ringkasan temuan EDA

Sel berikut mencetak ringkasan otomatis. Gunakan untuk poin **A.3f** (temuan yang memengaruhi preprocessing).

In [ ]:
print('RINGKASAN TEMUAN EDA')
print(f"1. Total {len(df)} file ({RESULTS['dataset_raw']['total_train']} train, {RESULTS['dataset_raw']['total_test']} test), 6 kelas; {len(invalid)} file rusak.")
print(f"2. Kelas tidak seimbang: rasio {RESULTS['eda']['imbalance_ratio_train']:.2f}x pada train -> dipakai class weight.")
print(f"3. Resolusi sangat bervariasi ({RESULTS['eda']['res_min']} s.d. {RESULTS['eda']['res_max']}) -> semua gambar di-resize ke {IMG_SIZE}x{IMG_SIZE}.")
print(f"4. Duplikat: {dup_stats['duplikat_di_dalam_train']} di train, {dup_stats['gambar_train_yang_juga_ada_di_test']} gambar train juga ada di test -> dihapus dari train untuk mencegah kebocoran.")
print('5. Kecerahan dan latar belakang bervariasi antarkelas -> augmentasi kontras, rotasi, zoom, dan flip.')
print('6. Cek contoh di 3.2: banyak gambar tampak berupa versi rotasi/flip dari foto yang sama (near-duplicate), sehingga akurasi test bisa sedikit optimistis.')

## 4. Preprocessing (Laporan A.4)

### 4.1 Pembersihan data
1. Buang file rusak.
2. Buang gambar yang muncul dengan label berbeda (label tidak konsisten).
3. Buang duplikat persis di dalam train (sisakan satu).
4. Buang gambar train yang identik dengan gambar test (mencegah kebocoran data).

Folder **test tidak diubah** (selain file rusak) agar hasil tetap sebanding dengan penelitian lain yang memakai dataset ini.

In [ ]:
steps = [('Train awal (file valid)', len(train_v))]
train_clean = train_v[~train_v['md5'].isin(conflict_md5)]
steps.append(('Setelah buang label tidak konsisten', len(train_clean)))
train_clean = train_clean.drop_duplicates('md5', keep='first')
steps.append(('Setelah buang duplikat di train', len(train_clean)))
train_clean = train_clean[~train_clean['md5'].isin(test_md5)]
steps.append(('Setelah buang yang identik dengan test', len(train_clean)))
test_df = test_v.copy().reset_index(drop=True)

clean_table = pd.DataFrame(steps, columns=['Tahap', 'Jumlah gambar train'])
print(clean_table.to_string(index=False))
print('Test (tidak diubah):', len(test_df))
RESULTS['cleaning'] = {s: int(n) for s, n in steps}

### 4.2 Pembagian data

Folder `train` dibagi menjadi **train** dan **validation** secara *stratified* (proporsi kelas sama). Folder `test` hanya dipakai satu kali di akhir untuk evaluasi.

In [ ]:
train_df, val_df = train_test_split(train_clean, test_size=VAL_SIZE, stratify=train_clean['label'],
                                    random_state=SEED)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)
assert not (set(train_df['md5']) & set(val_df['md5'])), 'Ada kebocoran train-val!'
assert not (set(train_df['md5']) & set(test_df['md5'])), 'Ada kebocoran train-test!'

split_table = pd.DataFrame({'train': train_df['label'].value_counts(),
                            'validation': val_df['label'].value_counts(),
                            'test': test_df['label'].value_counts()}).loc[CLASSES]
split_table.loc['TOTAL'] = split_table.sum()
total_all = split_table.loc['TOTAL'].sum()
print(split_table.to_string())
print('\nProporsi:', {k: f'{split_table.loc["TOTAL", k] / total_all:.1%}' for k in split_table.columns})

for name, frame in [('train', train_df), ('val', val_df), ('test', test_df)]:
    frame[['path', 'label', 'md5']].to_csv(os.path.join(RES_DIR, f'split_{name}.csv'), index=False)
RESULTS['split'] = {k: int(split_table.loc['TOTAL', k]) for k in split_table.columns}
RESULTS['split_per_class'] = split_table.drop(index='TOTAL').astype(int).to_dict(orient='index')

### 4.3 Pipeline data (resize, normalisasi) dan class weight

- **Resize** ke 224×224 (ukuran input MobileNetV2).
- **Normalisasi** piksel dari [0, 255] ke [-1, 1] dilakukan **di dalam model** (layer `Rescaling`), sama dengan `preprocess_input` MobileNetV2. Keuntungannya, aplikasi nanti cukup mengirim gambar RGB 224×224 tanpa preprocessing tambahan.
- **Class weight** (`balanced`) untuk mengimbangi kelas yang jumlahnya lebih sedikit.

In [ ]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)

def make_ds(frame, training=False):
    labels = frame['label'].map(CLASS_TO_IDX).values.astype('int32')
    ds = tf.data.Dataset.from_tensor_slices((frame['path'].values, labels))
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES),
                          y=train_df['label'].map(CLASS_TO_IDX).values)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
print('Class weight:', {CLASSES[i]: round(w, 3) for i, w in CLASS_WEIGHT.items()})
RESULTS['class_weight'] = {CLASSES[i]: w for i, w in CLASS_WEIGHT.items()}

### 4.4 Augmentasi data

Augmentasi hanya aktif saat training (otomatis nonaktif saat evaluasi dan prediksi):

| Jenis | Parameter | Alasan |
|---|---|---|
| RandomFlip | horizontal & vertical | posisi buah di foto bebas |
| RandomRotation | ±15% putaran (±54°) | buah difoto dari berbagai sudut |
| RandomZoom | ±15% | jarak kamera berbeda |
| RandomContrast | ±15% | variasi pencahayaan |

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name='augmentation')

sample_img = load_image(train_df.iloc[0]['path']).numpy()
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(sample_img); axes[0].set_title('Asli'); axes[0].axis('off')
for k in range(1, 6):
    aug = data_augmentation(tf.cast(sample_img[None], tf.float32), training=True)[0].numpy()
    axes[k].imshow(np.clip(aug, 0, 255).astype('uint8')); axes[k].set_title(f'Augmentasi {k}'); axes[k].axis('off')
plt.tight_layout(); savefig('04_contoh_augmentasi.png'); plt.show()

## 5. Model (Laporan A.5)

**Arsitektur:** Input 224×224×3 → Augmentasi → Rescaling [-1, 1] → **MobileNetV2** (bobot ImageNet, tanpa top) → GlobalAveragePooling2D → Dropout(0.3) → Dense(6, softmax, L2 1e-4)

**Strategi pelatihan dua tahap:**
1. **Tahap 1 (feature extraction):** MobileNetV2 dibekukan, hanya classifier head yang dilatih (Adam, lr 1e-3).
2. **Tahap 2 (fine-tuning):** layer MobileNetV2 mulai indeks `FINE_TUNE_AT` dibuka dan dilatih dengan learning rate kecil (1e-5) agar bobot ImageNet tidak rusak. Layer BatchNormalization tetap dibekukan.

**Alasan memilih MobileNetV2:** ringan (sekitar 2,3 juta parameter backbone), cepat untuk inferensi di aplikasi web/CPU, dan sudah terbukti baik untuk klasifikasi citra dengan transfer learning pada dataset berukuran sedang.

In [ ]:
def build_model():
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=WEIGHTS)
    base.trainable = False
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name='rescale')(x)
    x = base(x, training=False)            # BatchNorm selalu mode inferensi
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(DROPOUT, name='dropout')(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax',
                           kernel_regularizer=keras.regularizers.l2(L2_REG), name='pred')(x)
    return keras.Model(inputs, outputs, name='fruit_mobilenetv2'), base

model, base = build_model()

def count_params(m):
    tr = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    nt = int(sum(np.prod(w.shape) for w in m.non_trainable_weights))
    return tr, nt

model.summary()
arch = pd.DataFrame([{'Layer': l.name, 'Jenis': l.__class__.__name__,
                      'Output shape': str(tuple(l.output.shape)), 'Parameter': l.count_params()}
                     for l in model.layers])
print(arch.to_string(index=False))
arch.to_csv(os.path.join(RES_DIR, 'arsitektur_model.csv'), index=False)
tr, nt = count_params(model)
print(f'\nTahap 1 -> trainable: {tr:,} | non-trainable: {nt:,} | total: {tr+nt:,}')
print('Jumlah layer di MobileNetV2:', len(base.layers))
RESULTS['model'] = {'total_params': tr + nt, 'trainable_params_stage1': tr, 'base_layers': len(base.layers)}

try:
    keras.utils.plot_model(model, to_file=os.path.join(FIG_DIR, '05_arsitektur_model.png'),
                           show_shapes=True, show_layer_names=True, dpi=120)
    print('Diagram arsitektur disimpan ke figures/05_arsitektur_model.png')
except Exception as e:
    print('plot_model tidak tersedia (butuh pydot + graphviz):', e)

In [ ]:
def make_callbacks(tag):
    return [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1),
        keras.callbacks.ModelCheckpoint(os.path.join(MODEL_DIR, f'best_{tag}.keras'), monitor='val_loss',
                                        save_best_only=True),
        keras.callbacks.CSVLogger(os.path.join(RES_DIR, f'history_{tag}.csv')),
    ]

def get_preds(m, ds):
    probs = m.predict(ds, verbose=0)
    y = np.concatenate([yb.numpy() for _, yb in ds])
    return y, probs

def summarize(y, probs):
    pred = probs.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    _, _, fw, _ = precision_recall_fscore_support(y, pred, average='weighted', zero_division=0)
    return {'accuracy': float(accuracy_score(y, pred)), 'precision_macro': float(p),
            'recall_macro': float(r), 'f1_macro': float(f), 'f1_weighted': float(fw)}

### 5.1 Tahap 1: melatih classifier head (backbone dibekukan)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(LR_HEAD), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
t0 = time.time()
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                  class_weight=CLASS_WEIGHT, callbacks=make_callbacks('stage1'))
time_stage1 = time.time() - t0

y_val, p_val_A = get_preds(model, val_ds)
y_test, p_test_A = get_preds(model, test_ds)
RESULTS['skenario_A_feature_extraction'] = {
    'val': summarize(y_val, p_val_A), 'test': summarize(y_test, p_test_A),
    'epochs_run': len(hist1.history['loss']), 'train_time_s': round(time_stage1, 1),
    'trainable_params': count_params(model)[0]}
print(json.dumps(RESULTS['skenario_A_feature_extraction'], indent=2))

### 5.2 Tahap 2: fine-tuning

In [ ]:
base.trainable = True
for layer in base.layers[:FINE_TUNE_AT]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

tr, nt = count_params(model)
print(f'Tahap 2 -> trainable: {tr:,} | non-trainable: {nt:,}')
RESULTS['model']['trainable_params_stage2'] = tr

model.compile(optimizer=keras.optimizers.Adam(LR_FT), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
start_epoch = len(hist1.history['loss'])
t0 = time.time()
hist2 = model.fit(train_ds, validation_data=val_ds, initial_epoch=start_epoch,
                  epochs=start_epoch + EPOCHS_FT, class_weight=CLASS_WEIGHT, callbacks=make_callbacks('stage2'))
time_stage2 = time.time() - t0

y_val, p_val = get_preds(model, val_ds)
y_test, p_test = get_preds(model, test_ds)
RESULTS['skenario_B_fine_tuning'] = {
    'val': summarize(y_val, p_val), 'test': summarize(y_test, p_test),
    'epochs_run': len(hist2.history['loss']), 'train_time_s': round(time_stage1 + time_stage2, 1),
    'trainable_params': tr}
print(json.dumps(RESULTS['skenario_B_fine_tuning'], indent=2))

### 5.3 Grafik loss dan akurasi (training vs validation)

In [ ]:
H = {k: hist1.history[k] + hist2.history[k] for k in ['loss', 'val_loss', 'accuracy', 'val_accuracy']}
ep = np.arange(1, len(H['loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for a, key, title in [(ax[0], 'loss', 'Loss'), (ax[1], 'accuracy', 'Akurasi')]:
    a.plot(ep, H[key], 'o-', label='Training')
    a.plot(ep, H['val_' + key], 's-', label='Validation')
    a.axvline(start_epoch + 0.5, color='gray', ls='--', label='Mulai fine-tuning')
    a.set_title(f'{title} per epoch'); a.set_xlabel('Epoch'); a.legend()
plt.tight_layout(); savefig('06_kurva_loss_akurasi.png'); plt.show()
pd.DataFrame(H, index=ep).to_csv(os.path.join(RES_DIR, 'history_gabungan.csv'))

## 6. Evaluasi pada data test (Laporan A.6)

### 6.1 Metrik dan classification report

In [ ]:
y_pred = p_test.argmax(1)
report_txt = classification_report(y_test, y_pred, target_names=CLASSES, digits=4)
print(report_txt)
report_df = pd.DataFrame(classification_report(y_test, y_pred, target_names=CLASSES, digits=4,
                                               output_dict=True)).T
report_df.to_csv(os.path.join(RES_DIR, 'classification_report_test.csv'))
RESULTS['test_final'] = RESULTS['skenario_B_fine_tuning']['test']
RESULTS['test_per_class'] = {c: {k: float(report_df.loc[c, k]) for k in ['precision', 'recall', 'f1-score', 'support']}
                             for c in CLASSES}

### 6.2 Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=range(NUM_CLASSES))
cm_norm = cm / cm.sum(axis=1, keepdims=True)
names = [DISPLAY_NAMES.get(c, c) for c in CLASSES]
fig, ax = plt.subplots(1, 2, figsize=(17, 6.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=names, yticklabels=names, ax=ax[0])
ax[0].set_title('Confusion matrix (jumlah)')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=names, yticklabels=names, ax=ax[1])
ax[1].set_title('Confusion matrix (dinormalisasi per kelas asli)')
for a in ax:
    a.set_xlabel('Prediksi'); a.set_ylabel('Label asli'); a.tick_params(axis='x', rotation=45)
plt.tight_layout(); savefig('07_confusion_matrix.png'); plt.show()
RESULTS['confusion_matrix'] = cm.tolist()

### 6.3 Analisis kesalahan

In [ ]:
res = test_df[['path', 'label', 'fruit', 'condition']].copy()
res['pred'] = [CLASSES[i] for i in y_pred]
res['conf'] = p_test.max(1)
res['pred_fruit'] = res['pred'].str.replace('fresh', '').str.replace('rotten', '')
res['pred_condition'] = np.where(res['pred'].str.startswith('fresh'), 'segar', 'busuk')
wrong = res[res['label'] != res['pred']].sort_values('conf', ascending=False)

err_type = {
    'total_salah': int(len(wrong)),
    'salah_jenis_buah': int((wrong['fruit'] != wrong['pred_fruit']).sum()),
    'salah_kondisi_saja (segar<->busuk)': int(((wrong['fruit'] == wrong['pred_fruit']) &
                                               (wrong['condition'] != wrong['pred_condition'])).sum()),
}
print('Jenis kesalahan:', err_type)
print('\nPasangan kesalahan terbanyak (asli -> prediksi):')
pairs = (wrong['label'] + ' -> ' + wrong['pred']).value_counts().head(10)
print(pairs.to_string() if len(pairs) else 'Tidak ada kesalahan')
RESULTS['error_analysis'] = {**err_type, 'top_pairs': pairs.to_dict()}

n = min(12, len(wrong))
if n:
    cols = 6; rws = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rws, cols, figsize=(cols * 2.8, rws * 3.2))
    for a in np.ravel(axes): a.axis('off')
    for a, (_, r) in zip(np.ravel(axes), wrong.head(n).iterrows()):
        a.imshow(load_image(r['path']).numpy())
        a.set_title(f"Asli: {DISPLAY_NAMES[r['label']]}\nPred: {DISPLAY_NAMES[r['pred']]} ({r['conf']:.0%})",
                    fontsize=8, color='crimson')
    plt.suptitle('Contoh prediksi salah (diurutkan dari confidence tertinggi)')
    plt.tight_layout(); savefig('08_contoh_salah_prediksi.png'); plt.show()

### 6.4 Indikasi overfitting / underfitting

In [ ]:
final_train_acc = H['accuracy'][int(np.argmin(H['val_loss']))]
final_val_acc = H['val_accuracy'][int(np.argmin(H['val_loss']))]
test_acc = RESULTS['test_final']['accuracy']
gap = final_train_acc - final_val_acc
print(f'Akurasi train (epoch terbaik) : {final_train_acc:.4f}')
print(f'Akurasi validation            : {final_val_acc:.4f}')
print(f'Akurasi test                  : {test_acc:.4f}')
print(f'Selisih train - validation    : {gap:+.4f}')
if gap > 0.05:
    verdict = 'Ada indikasi overfitting (selisih train-val > 5%).'
elif final_train_acc < 0.85:
    verdict = 'Ada indikasi underfitting (akurasi train masih rendah).'
else:
    verdict = 'Tidak ada indikasi overfitting/underfitting yang signifikan.'
print('Kesimpulan:', verdict)
print('Catatan: akurasi train diukur dengan augmentasi + dropout aktif, sehingga bisa sedikit lebih rendah dari validation.')
RESULTS['fit_check'] = {'train_acc_best': float(final_train_acc), 'val_acc_best': float(final_val_acc),
                        'gap': float(gap), 'verdict': verdict}

### 6.5 Perbandingan skenario: feature extraction saja vs + fine-tuning

In [ ]:
A, B = RESULTS['skenario_A_feature_extraction'], RESULTS['skenario_B_fine_tuning']
comp = pd.DataFrame({
    'Skenario A: backbone beku': {**{k: round(v, 4) for k, v in A['test'].items()},
                                  'trainable_params': A['trainable_params'], 'waktu_latih_detik': A['train_time_s']},
    'Skenario B: + fine-tuning': {**{k: round(v, 4) for k, v in B['test'].items()},
                                  'trainable_params': B['trainable_params'], 'waktu_latih_detik': B['train_time_s']},
}).T
print(comp.to_string())
comp.to_csv(os.path.join(RES_DIR, 'perbandingan_skenario.csv'))

metrics_cols = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
comp[metrics_cols].astype(float).T.plot(kind='bar', figsize=(9, 4.5), ylim=(max(0, comp[metrics_cols].astype(float).values.min() - 0.05), 1.0))
plt.title('Perbandingan metrik test antarskenario'); plt.xticks(rotation=0); plt.legend(loc='lower right')
plt.tight_layout(); savefig('09_perbandingan_skenario.png'); plt.show()

## 7. Fitur unik 1: Grad-CAM (Laporan A.5e)

**Apa:** Grad-CAM menghasilkan *heatmap* yang menunjukkan bagian gambar yang paling memengaruhi keputusan model.

**Mengapa dibutuhkan:** CNN bersifat *black box*. Untuk klasifikasi segar/busuk, kita perlu memastikan model benar-benar melihat **bercak busuk, warna, dan tekstur kulit buah**, bukan latar belakang foto.

**Implementasi:** gradien skor kelas terhadap feature map terakhir MobileNetV2 (`out_relu`, 7×7×1280) dirata-ratakan menjadi bobot per kanal, lalu dijumlahkan berbobot dan dilewatkan ReLU.

In [ ]:
rescale_layer, gap_layer, pred_layer = model.get_layer('rescale'), model.get_layer('gap'), model.get_layer('pred')

def grad_cam(img_uint8, class_idx=None):
    x = tf.cast(img_uint8[None], tf.float32)
    with tf.GradientTape() as tape:
        conv = base(rescale_layer(x), training=False)
        tape.watch(conv)
        probs = pred_layer(gap_layer(conv))
        idx = int(tf.argmax(probs[0])) if class_idx is None else class_idx
        score = probs[:, idx]
    grads = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv * weights[:, None, None, :], axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy(), probs[0].numpy()

def overlay(img_uint8, cam, alpha=0.45):
    heat = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE)).numpy()[..., 0]
    color = plt.get_cmap('jet')(heat)[..., :3]
    return np.clip((1 - alpha) * img_uint8 / 255.0 + alpha * color, 0, 1)

correct = res[res['label'] == res['pred']]
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(12, NUM_CLASSES * 3))
for i, c in enumerate(CLASSES):
    pool = correct[correct['label'] == c]
    picks = pool.sample(min(2, len(pool)), random_state=SEED) if len(pool) else pool
    for j in range(2):
        a1, a2 = axes[i, j * 2], axes[i, j * 2 + 1]
        a1.axis('off'); a2.axis('off')
        if j >= len(picks):
            continue
        r = picks.iloc[j]
        img = load_image(r['path']).numpy()
        cam, pr = grad_cam(img)
        a1.imshow(img); a1.set_title(DISPLAY_NAMES[c], fontsize=9)
        a2.imshow(overlay(img, cam)); a2.set_title(f'Grad-CAM ({pr.max():.0%})', fontsize=9)
plt.suptitle('Grad-CAM pada prediksi benar (2 contoh per kelas)', y=1.0)
plt.tight_layout(); savefig('10_gradcam_prediksi_benar.png'); plt.show()

if len(wrong):
    n = min(4, len(wrong))
    fig, axes = plt.subplots(n, 2, figsize=(6.5, n * 3.1))
    axes = np.atleast_2d(axes)
    for k, (_, r) in enumerate(wrong.head(n).iterrows()):
        img = load_image(r['path']).numpy()
        cam, pr = grad_cam(img)
        axes[k, 0].imshow(img); axes[k, 0].set_title(f"Asli: {DISPLAY_NAMES[r['label']]}", fontsize=9)
        axes[k, 1].imshow(overlay(img, cam)); axes[k, 1].set_title(f"Pred: {DISPLAY_NAMES[r['pred']]} ({pr.max():.0%})", fontsize=9, color='crimson')
        axes[k, 0].axis('off'); axes[k, 1].axis('off')
    plt.suptitle('Grad-CAM pada prediksi salah', y=1.0)
    plt.tight_layout(); savefig('11_gradcam_prediksi_salah.png'); plt.show()

## 8. Fitur unik 2: Confidence threshold dan deteksi input di luar domain (Laporan A.5e)

**Apa:** prediksi hanya diterima jika *confidence* (probabilitas softmax tertinggi) ≥ ambang batas. Jika di bawahnya, aplikasi menjawab **"tidak yakin / bukan buah yang dikenali"**.

**Mengapa dibutuhkan:** softmax selalu memilih salah satu dari 6 kelas, walaupun pengguna mengunggah foto yang bukan apel, pisang, atau jeruk. Threshold membuat aplikasi lebih jujur dan aman: prediksi yang meragukan ditolak.

**Implementasi:**
1. Ambang ditentukan dari **data validation** (bukan test, agar tidak bocor): dipilih nilai yang tetap mempertahankan 95% prediksi benar di validation.
2. Dampaknya diukur pada data test (akurasi pada prediksi yang diterima, coverage, kesalahan yang berhasil ditolak).
3. Diuji pada gambar **di luar domain** (noise, warna polos, gradasi, pola), plus folder opsional `OOD_DIR` berisi foto non-buah.

In [ ]:
def selective_table(y, probs, thresholds):
    conf, pred = probs.max(1), probs.argmax(1)
    ok = pred == y
    rows = []
    for t in thresholds:
        acc_mask = conf >= t
        rows.append({'threshold': t, 'coverage': acc_mask.mean(),
                     'akurasi_yang_diterima': ok[acc_mask].mean() if acc_mask.any() else np.nan,
                     'ditolak': int((~acc_mask).sum()),
                     'kesalahan_tertolak': int((~ok & ~acc_mask).sum()), 'total_kesalahan': int((~ok).sum()),
                     'benar_tapi_tertolak': int((ok & ~acc_mask).sum())})
    return pd.DataFrame(rows)

val_conf_correct = p_val.max(1)[p_val.argmax(1) == y_val]
THRESHOLD = (float(np.clip(np.floor(np.percentile(val_conf_correct, 5) * 100) / 100, 0.5, 0.95))
             if len(val_conf_correct) else 0.70)
print(f'Threshold terpilih (persentil 5 confidence prediksi benar di validation): {THRESHOLD:.2f}')

thresholds = np.round(np.arange(0.40, 1.00, 0.05), 2)
tbl_test = selective_table(y_test, p_test, thresholds)
print('\nEfek berbagai threshold pada data test:')
print(tbl_test.round(4).to_string(index=False))
tbl_test.to_csv(os.path.join(RES_DIR, 'threshold_sweep_test.csv'), index=False)

at = selective_table(y_test, p_test, [THRESHOLD]).iloc[0]
RESULTS['threshold'] = {'value': THRESHOLD, 'test_coverage': float(at['coverage']),
                        'test_acc_accepted': float(at['akurasi_yang_diterima']),
                        'test_acc_no_threshold': RESULTS['test_final']['accuracy'],
                        'errors_rejected': int(at['kesalahan_tertolak']), 'total_errors': int(at['total_kesalahan']),
                        'correct_rejected': int(at['benar_tapi_tertolak'])}
print('\nPada threshold terpilih:', json.dumps(RESULTS['threshold'], indent=2))

conf_t, ok_t = p_test.max(1), p_test.argmax(1) == y_test
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(tbl_test['coverage'], tbl_test['akurasi_yang_diterima'], 'o-')
for _, r in tbl_test.iterrows():
    ax[0].annotate(f"{r['threshold']:.2f}", (r['coverage'], r['akurasi_yang_diterima']), fontsize=7, xytext=(3, 3), textcoords='offset points')
ax[0].set_xlabel('Coverage (proporsi prediksi diterima)'); ax[0].set_ylabel('Akurasi pada prediksi diterima')
ax[0].set_title('Trade-off coverage vs akurasi (test)')
bins = np.linspace(0, 1, 41)
ax[1].hist(conf_t[ok_t], bins=bins, alpha=0.7, label='Prediksi benar', log=True)
ax[1].hist(conf_t[~ok_t], bins=bins, alpha=0.7, label='Prediksi salah', log=True)
ax[1].axvline(THRESHOLD, color='k', ls='--', label=f'Threshold {THRESHOLD:.2f}')
ax[1].set_title('Distribusi confidence pada test (skala log)'); ax[1].set_xlabel('Confidence'); ax[1].legend()
plt.tight_layout(); savefig('12_threshold_tradeoff.png'); plt.show()

In [ ]:
rng = np.random.default_rng(SEED)
def synthetic_ood(n_each=40):
    imgs, kinds = [], []
    for _ in range(n_each):
        imgs.append(rng.integers(0, 256, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)); kinds.append('Noise acak')
        small = rng.integers(0, 256, (8, 8, 3), dtype=np.uint8)
        imgs.append(np.asarray(Image.fromarray(small).resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))); kinds.append('Bercak warna (blur noise)')
        imgs.append(np.full((IMG_SIZE, IMG_SIZE, 3), rng.integers(0, 256, 3), dtype=np.uint8)); kinds.append('Warna polos')
        c1, c2 = rng.integers(0, 256, 3), rng.integers(0, 256, 3)
        g = np.linspace(0, 1, IMG_SIZE)[None, :, None]
        imgs.append(np.repeat((c1 * (1 - g) + c2 * g), IMG_SIZE, axis=0).astype(np.uint8)); kinds.append('Gradasi')
        s = int(rng.integers(8, 56))
        yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
        mask = ((yy // s + xx // s) % 2).astype(bool)
        pat = np.empty((IMG_SIZE, IMG_SIZE, 3), np.uint8); pat[mask] = rng.integers(0, 256, 3); pat[~mask] = rng.integers(0, 256, 3)
        imgs.append(pat); kinds.append('Pola kotak-kotak')
    return np.stack(imgs), kinds

ood_imgs, ood_kinds = synthetic_ood()
if OOD_DIR and os.path.isdir(OOD_DIR):
    extra = [p for p in glob.glob(os.path.join(OOD_DIR, '**', '*'), recursive=True)
             if p.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))][:500]
    if extra:
        ood_imgs = np.concatenate([ood_imgs, np.stack([load_image(p).numpy() for p in extra])])
        ood_kinds += ['Foto non-buah (OOD_DIR)'] * len(extra)
        print('Menambahkan', len(extra), 'foto non-buah dari', OOD_DIR)

p_ood = model.predict(ood_imgs.astype('float32'), batch_size=BATCH_SIZE, verbose=0)
ood = pd.DataFrame({'jenis': ood_kinds, 'conf': p_ood.max(1), 'pred': [CLASSES[i] for i in p_ood.argmax(1)]})
ood['ditolak'] = ood['conf'] < THRESHOLD
per_kind = ood.groupby('jenis').agg(jumlah=('conf', 'size'), conf_rata2=('conf', 'mean'),
                                    persen_ditolak=('ditolak', 'mean'),
                                    prediksi_terbanyak=('pred', lambda s: s.value_counts().index[0]))
print(per_kind.round(3).to_string())

scores = np.concatenate([p_test.max(1), p_ood.max(1)])
is_in = np.concatenate([np.ones(len(p_test)), np.zeros(len(p_ood))])
auroc = roc_auc_score(is_in, scores)
print(f'\nAUROC (pemisahan test vs luar domain berdasarkan confidence): {auroc:.4f}')
print(f'Persentase gambar luar domain yang berhasil ditolak: {ood["ditolak"].mean():.1%}')
print(f'Persentase gambar test (buah asli) yang ditolak    : {1 - RESULTS["threshold"]["test_coverage"]:.1%}')
RESULTS['ood'] = {'n': int(len(ood)), 'rejected_rate': float(ood['ditolak'].mean()), 'auroc_msp': float(auroc),
                  'per_kind': per_kind.reset_index().to_dict(orient='records')}

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
ax[0].hist(p_test.max(1), bins=bins, alpha=0.6, density=True, label='Test (buah)')
ax[0].hist(p_ood.max(1), bins=bins, alpha=0.6, density=True, label='Luar domain')
ax[0].axvline(THRESHOLD, color='k', ls='--', label=f'Threshold {THRESHOLD:.2f}')
ax[0].set_title('Confidence: data buah vs luar domain'); ax[0].set_xlabel('Confidence'); ax[0].legend()
(per_kind['persen_ditolak'] * 100).plot(kind='barh', ax=ax[1], color='#55A868')
ax[1].set_xlabel('% ditolak'); ax[1].set_title('Tingkat penolakan per jenis input luar domain'); ax[1].set_xlim(0, 100)
plt.tight_layout(); savefig('13_ood_confidence.png'); plt.show()

uniq = ood.drop_duplicates('jenis').index
fig, axes = plt.subplots(1, len(uniq), figsize=(len(uniq) * 3, 3.3))
for a, i in zip(np.atleast_1d(axes), uniq):
    a.imshow(ood_imgs[i]); a.axis('off')
    status = 'DITOLAK' if ood.loc[i, 'ditolak'] else DISPLAY_NAMES[ood.loc[i, 'pred']]
    a.set_title(f"{ood.loc[i, 'jenis']}\n{status} ({ood.loc[i, 'conf']:.0%})", fontsize=8,
                color='green' if ood.loc[i, 'ditolak'] else 'crimson')
plt.tight_layout(); savefig('14_contoh_ood.png'); plt.show()

**Tabel perbandingan tanpa vs dengan fitur unik 2** (untuk laporan A.6d). Jika tingkat penolakan untuk jenis input tertentu masih rendah, tuliskan sebagai keterbatasan: confidence softmax tidak selalu cukup untuk mendeteksi input luar domain, dan ini dapat dikembangkan pada penelitian berikutnya.

In [ ]:
th = RESULTS['threshold']
comp2 = pd.DataFrame({
    'Tanpa threshold': {'akurasi_test': th['test_acc_no_threshold'], 'coverage_test': 1.0,
                        'kesalahan_test_tertolak': 0, 'input_luar_domain_ditolak': 0.0},
    f'Dengan threshold {THRESHOLD:.2f}': {'akurasi_test': th['test_acc_accepted'], 'coverage_test': th['test_coverage'],
                                         'kesalahan_test_tertolak': f"{th['errors_rejected']}/{th['total_errors']}",
                                         'input_luar_domain_ditolak': RESULTS['ood']['rejected_rate']},
}).T
print(comp2.to_string())
comp2.to_csv(os.path.join(RES_DIR, 'perbandingan_threshold.csv'))

## 9. Simpan model, konfigurasi, dan hasil

In [ ]:
MODEL_PATH = os.path.join(MODEL_DIR, 'fruit_mobilenetv2.keras')
model.save(MODEL_PATH)
app_config = {
    'model_file': 'fruit_mobilenetv2.keras', 'img_size': IMG_SIZE, 'classes': CLASSES,
    'display_names': DISPLAY_NAMES, 'confidence_threshold': THRESHOLD,
    'input': 'Gambar RGB di-resize ke 224x224, nilai piksel 0-255 (float32). Normalisasi ke [-1,1] sudah ada di dalam model.',
    'tensorflow_version': tf.__version__, 'keras_version': keras.__version__,
}
with open(os.path.join(MODEL_DIR, 'config.json'), 'w') as f:
    json.dump(app_config, f, indent=2)

def to_builtin(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
with open(os.path.join(RES_DIR, 'metrics.json'), 'w') as f:
    json.dump(RESULTS, f, indent=2, default=to_builtin)

# Uji: muat ulang model dari file dan pastikan prediksinya sama
loaded = keras.models.load_model(MODEL_PATH)
xb, _ = next(iter(test_ds))
diff = np.abs(loaded.predict(xb, verbose=0) - model.predict(xb, verbose=0)).max()
print('Selisih maksimum prediksi model asli vs model dimuat ulang:', diff)
print('Ukuran file model: %.1f MB' % (os.path.getsize(MODEL_PATH) / 1e6))

In [ ]:
def predict_image(path, mdl=loaded, threshold=THRESHOLD):
    """Fungsi inferensi yang sama dengan yang akan dipakai di aplikasi."""
    img = load_image(path).numpy().astype('float32')[None]
    probs = mdl.predict(img, verbose=0)[0]
    idx = int(probs.argmax())
    conf = float(probs[idx])
    label = DISPLAY_NAMES[CLASSES[idx]] if conf >= threshold else 'Tidak yakin / bukan buah yang dikenali'
    return {'label': label, 'confidence': round(conf, 4),
            'probabilitas': {DISPLAY_NAMES[c]: round(float(p), 4) for c, p in zip(CLASSES, probs)}}

for _, r in test_df.sample(3, random_state=1).iterrows():
    print(DISPLAY_NAMES[r['label']], '->', predict_image(r['path']))

In [ ]:
zip_path = os.path.join(OUT_DIR, 'hasil_proyek.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in (FIG_DIR, RES_DIR, MODEL_DIR):
        for p in glob.glob(os.path.join(folder, '*')):
            if os.path.basename(p).startswith('best_'):
                continue
            z.write(p, os.path.relpath(p, OUT_DIR))
print('Semua hasil dikemas di:', zip_path, '(%.1f MB)' % (os.path.getsize(zip_path) / 1e6))
print('Unduh lewat panel Output di kanan (atau tab Output setelah Save Version).')

## 10. Teknologi yang digunakan (Laporan A.7)

In [ ]:
env = {'Python': platform.python_version(), 'TensorFlow': tf.__version__, 'Keras': keras.__version__,
       'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
       'Matplotlib': matplotlib.__version__, 'Seaborn': sns.__version__, 'Pillow': PIL.__version__,
       'OS': platform.platform()}
try:
    env['GPU'] = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                                capture_output=True, text=True).stdout.strip().replace('\n', ' | ') or 'Tidak ada'
except Exception:
    env['GPU'] = 'Tidak ada'
for k, val in env.items():
    print(f'{k:13s}: {val}')
RESULTS['environment'] = env
with open(os.path.join(RES_DIR, 'metrics.json'), 'w') as f:
    json.dump(RESULTS, f, indent=2, default=to_builtin)